## Model Training

####  Import Data and Required Packages
##### Importing Pandas, Numpy, Matplotlib, Seaborn and Warings Library etc.

In [16]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    KFold
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    AdaBoostRegressor,
    GradientBoostingRegressor
)

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings("ignore")

#### Import the CSV Data as Pandas DataFrame

In [17]:
df = pd.read_csv('data/stud.csv')
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


#### X/y + Train/Test Split

In [18]:
X = df.drop(columns=["math_score"])
y = df["math_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (800, 7)
X_test : (200, 7)
y_train: (800,)
y_test : (200,)


#### Preprocessing Pipeline

In [19]:
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features)
    ]
)

#### Evaluation Function

In [20]:
def evaluate_model(y_true, y_pred):
    
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }

#### Reusable Model Training Function

In [21]:
def train_model(model, X_train, X_test, y_train, y_test):
    
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )
    
    pipeline.fit(X_train, y_train)
    
    train_pred = pipeline.predict(X_train)
    test_pred = pipeline.predict(X_test)
    
    train_metrics = evaluate_model(y_train, train_pred)
    test_metrics = evaluate_model(y_test, test_pred)
    
    return pipeline, train_metrics, test_metrics

#### Models Dictionary

In [22]:
models = {
    "Linear Regression": LinearRegression(),
    
    "Lasso": Lasso(),
    
    "Ridge": Ridge(),
    
    "K-Neighbors Regressor": KNeighborsRegressor(),
    
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    
    "Random Forest Regressor": RandomForestRegressor(
        random_state=42
    ),
    
    "XGBRegressor": XGBRegressor(
        random_state=42,
        objective="reg:squarederror"
    ),
    
    "CatBoosting Regressor": CatBoostRegressor(
        verbose=False,
        random_state=42
    ),
    
    "AdaBoost Regressor": AdaBoostRegressor(
        random_state=42
    )
}

#### All Models Automatically Train + Evaluate

In [23]:
results = []
trained_models = {}

for name, model in models.items():
    
    pipeline, train_metrics, test_metrics = train_model(
        model,
        X_train,
        X_test,
        y_train,
        y_test
    )
    
    trained_models[name] = pipeline
    
    results.append({
        "Model": name,
        "Train MAE": train_metrics["MAE"],
        "Train RMSE": train_metrics["RMSE"],
        "Train R2": train_metrics["R2"],
        "Test MAE": test_metrics["MAE"],
        "Test RMSE": test_metrics["RMSE"],
        "Test R2": test_metrics["R2"]
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    by="Test R2",
    ascending=False
)

,Model,Train MAE,Train RMSE,Train R2,Test MAE,Test RMSE,Test R2
2,Ridge,4.265005,5.323321,0.874304,4.211113,5.390418,0.880592
0,Linear Regression,4.266712,5.323051,0.874317,4.214763,5.393994,0.880433
8,AdaBoost Regressor,4.730278,5.780344,0.851795,4.681343,6.008278,0.851649
7,CatBoosting Regressor,2.402360,3.053013,0.958656,4.589369,6.033317,0.850410
5,Random Forest Regressor,1.824858,2.303510,0.976464,4.701467,6.033663,0.850393
1,Lasso,5.205260,6.592500,0.807223,5.155701,6.517328,0.825447
6,XGBRegressor,0.687467,1.007282,0.995500,5.103629,6.595750,0.821220
3,K-Neighbors Regressor,4.511250,5.700206,0.855876,5.592000,7.249386,0.784031
4,Decision Tree,0.018750,0.279508,0.999653,6.240000,7.840918,0.747348


#### Tuning Function

In [24]:
def tune_model(
    model,
    param_distributions,
    X_train,
    y_train,
    n_iter=20,
    cv=5
):
    
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )
    
    random_search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring="r2",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    
    random_search.fit(X_train, y_train)
    
    print("Best Parameters:")
    print(random_search.best_params_)
    
    print("\nBest CV R2:")
    print(random_search.best_score_)
    
    return random_search

#### XGBoost Tuning + Evaluation

In [25]:
xgb_params = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5, 7]
}


xgb_search = tune_model(
    model=XGBRegressor(
        random_state=42,
        objective="reg:squarederror"
    ),
    param_distributions=xgb_params,
    X_train=X_train,
    y_train=y_train,
    n_iter=30,
    cv=5
)


best_xgb = xgb_search.best_estimator_

xgb_train_pred = best_xgb.predict(X_train)
xgb_test_pred = best_xgb.predict(X_test)

xgb_train_metrics = evaluate_model(
    y_train,
    xgb_train_pred
)

xgb_test_metrics = evaluate_model(
    y_test,
    xgb_test_pred
)

print("XGBoost Tuned Model")
print("=" * 40)

print("\nTraining:")
for metric, value in xgb_train_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTesting:")
for metric, value in xgb_test_metrics.items():
    print(f"{metric}: {value:.4f}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters:
{'model__subsample': 0.9, 'model__n_estimators': 300, 'model__min_child_weight': 5, 'model__max_depth': 4, 'model__learning_rate': 0.03, 'model__colsample_bytree': 1.0}

Best CV R2:
0.8491501504691605
XGBoost Tuned Model

Training:
MAE: 3.3997
MSE: 18.0119
RMSE: 4.2440
R2: 0.9201

Testing:
MAE: 4.3640
MSE: 32.5956
RMSE: 5.7093
R2: 0.8660


#### Random Forest Tuning

In [26]:
rf_params = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 5, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", 1.0]
}


rf_search = tune_model(
    model=RandomForestRegressor(
        random_state=42
    ),
    param_distributions=rf_params,
    X_train=X_train,
    y_train=y_train,
    n_iter=30,
    cv=5
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters:
{'model__n_estimators': 200, 'model__min_samples_split': 5, 'model__min_samples_leaf': 4, 'model__max_features': 1.0, 'model__max_depth': None}

Best CV R2:
0.8401577919363703


#### CatBoost Tuning

In [27]:
cat_params = {
    "model__iterations": [100, 200, 300, 500],
    "model__depth": [4, 5, 6, 7, 8, 10],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "model__l2_leaf_reg": [1, 3, 5, 7, 10]
}

cat_search = tune_model(
    model=CatBoostRegressor(
        verbose=False,
        random_state=42
    ),
    param_distributions=cat_params,
    X_train=X_train,
    y_train=y_train,
    n_iter=30,
    cv=5
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters:
{'model__learning_rate': 0.03, 'model__l2_leaf_reg': 5, 'model__iterations': 500, 'model__depth': 4}

Best CV R2:
0.8606103762172861


#### Best 3 Models ka Final Comparison

In [28]:
tuned_models = {
    "XGBoost Tuned": xgb_search.best_estimator_,
    "Random Forest Tuned": rf_search.best_estimator_,
    "CatBoost Tuned": cat_search.best_estimator_
}


tuned_results = []

for name, model in tuned_models.items():
    
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    
    train_metrics = evaluate_model(
        y_train,
        train_pred
    )
    
    test_metrics = evaluate_model(
        y_test,
        test_pred
    )
    
    tuned_results.append({
        "Model": name,
        
        "Train MAE": train_metrics["MAE"],
        "Train RMSE": train_metrics["RMSE"],
        "Train R2": train_metrics["R2"],
        
        "Test MAE": test_metrics["MAE"],
        "Test RMSE": test_metrics["RMSE"],
        "Test R2": test_metrics["R2"]
    })

tuned_results_df = pd.DataFrame(tuned_results)

tuned_results_df.sort_values(
    by="Test R2",
    ascending=False
)

,Model,Train MAE,Train RMSE,Train R2,Test MAE,Test RMSE,Test R2
2,CatBoost Tuned,3.890193,4.838954,0.896138,4.274664,5.588469,0.871656
0,XGBoost Tuned,3.399661,4.244048,0.920106,4.364032,5.709252,0.866048
1,Random Forest Tuned,3.492008,4.335606,0.916621,4.544273,5.968487,0.853608


In [29]:
baseline_best = results_df.sort_values(
    by="Test R2",
    ascending=False
).head(3)

tuned_results_df = tuned_results_df.sort_values(
    by="Test R2",
    ascending=False
)

display(baseline_best)
display(tuned_results_df)


,Model,Train MAE,Train RMSE,Train R2,Test MAE,Test RMSE,Test R2
2,Ridge,4.265005,5.323321,0.874304,4.211113,5.390418,0.880592
0,Linear Regression,4.266712,5.323051,0.874317,4.214763,5.393994,0.880433
8,AdaBoost Regressor,4.730278,5.780344,0.851795,4.681343,6.008278,0.851649


,Model,Train MAE,Train RMSE,Train R2,Test MAE,Test RMSE,Test R2
2,CatBoost Tuned,3.890193,4.838954,0.896138,4.274664,5.588469,0.871656
0,XGBoost Tuned,3.399661,4.244048,0.920106,4.364032,5.709252,0.866048
1,Random Forest Tuned,3.492008,4.335606,0.916621,4.544273,5.968487,0.853608


In [30]:
import os
import json
import joblib

# Create folders
os.makedirs("ridge_info/learn", exist_ok=True)
os.makedirs("ridge_info/tmp", exist_ok=True)


# Train Ridge
final_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge())
    ]
)

final_model.fit(X_train, y_train)


# Predictions
train_pred = final_model.predict(X_train)
test_pred = final_model.predict(X_test)


# Evaluation
train_metrics = evaluate_model(y_train, train_pred)
test_metrics = evaluate_model(y_test, test_pred)


# Save model
joblib.dump(
    final_model,
    "ridge_info/ridge_model.pkl"
)


# Training JSON
training_info = {
    "model": "Ridge Regression",
    "status": "completed",
    "metrics": {
        "train": train_metrics,
        "test": test_metrics
    }
}

with open("ridge_info/ridge_training.json", "w") as f:
    json.dump(training_info, f, indent=4)


# Learn error
with open("ridge_info/learn_error.tsv", "w") as f:
    f.write("Metric\tValue\n")

    for metric, value in train_metrics.items():
        f.write(f"{metric}\t{value}\n")


# Time left
with open("ridge_info/time_left.tsv", "w") as f:
    f.write("Status\tValue\n")
    f.write("Training\tCompleted\n")


print("✅ Ridge training completed!")
print("✅ Ridge model saved!")
print("✅ ridge_info folder created!")

✅ Ridge training completed!
✅ Ridge model saved!
✅ ridge_info folder created!
